# 04 — Supervised Fine-Tuning (SFT) Baseline
**Goal**: Fine-tune the 1.3B base model on APPS problem-solution pairs using LoRA adapters. Produces the `./checkpoints/sft/final` adapter checkpoint to warm-start RL training (PPO & DPO).

---

## Step 1: Environment & Tokenizer Parallelism Configuration

In [ ]:
import os
import torch
from datasets import load_dataset
from src.models.loader import load_model_and_tokenizer
from src.training.sft import format_for_sft, run_sft_training

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment initialized!")

## Step 2: Load APPS Dataset & Format for SFT

In [ ]:
print("Loading APPS training dataset...")
apps = load_dataset('codeparrot/apps', split='train[:2000]', trust_remote_code=True)
apps_clean = apps.filter(lambda x: len(x['solutions']) > 0)
print(f"Loaded {len(apps_clean)} APPS problem-solution pairs.")

sample_formatted = format_for_sft(apps_clean[0])
print("\n--- Sample Formatted SFT Prompt ---")
print(sample_formatted['text'][:300] + "...")

## Step 3: Load Base Model & Run SFT Training

In [ ]:
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
print(f"Loading base model {MODEL_NAME} for SFT...")
model, tokenizer = load_model_and_tokenizer(model_name=MODEL_NAME, load_in_4bit=True, lora_r=16)

print("\nStarting SFT Training loop...")
trainer = run_sft_training(
    model=model,
    tokenizer=tokenizer,
    dataset=apps_clean,
    output_dir="./checkpoints/sft",
    num_epochs=3,
    per_device_batch_size=4,
    gradient_accumulation_steps=4, # Effective batch size = 16
    learning_rate=2e-5,
)

print("\nSFT Training completed successfully!")
print("Saved fine-tuned adapter to ./checkpoints/sft/final")